读取验证结果文件

In [ ]:
from utils import read_jsonl_gz
ds=read_jsonl_gz('../spec-gen/sample_results_old/Qwen3_32B.jsonl_results.jsonl.gz')

# 打印总共测试样例的数量
sum(
    len(d['test_cases_results'])
    for d in ds
)

In [ ]:
from random import choice
d=choice(ds)
choice(d['test_cases_results'])

统计所有题目pre and full对了多少

In [ ]:
def eval_spec_results(ds):
    all_results={}
    for d in ds:
        if len(d['test_cases_results'])>0:
            all_results[d['task_id']]=dict(
                positive_pre_result=True,
                negative_pre_result=True,
                positive_post_result=True,
                hard_negatives_post_result=True,
                random_negatives_post_result=True,
            )
        else:
            continue
        # 这个检查写的有问题
        for test_cases_result in d['test_cases_results']:
            # 有后续的ut
            if test_cases_result['test_case']['run_success']==True:
                
                # positive_pre_result应该为True
                if test_cases_result['pre_result']['passed']==False:
                    all_results[d['task_id']]['positive_pre_result']=False
                    
                # positive_post_result应该为True
                if test_cases_result['post_result']['passed']==False:
                    all_results[d['task_id']]['positive_post_result']=False
                    
                # hard_negatives_post_result应该为False
                if 'hard_negatives_post_result' in test_cases_result and test_cases_result['hard_negatives_post_result']['passed']==True:
                    all_results[d['task_id']]['hard_negatives_post_result']=False
                    
                # random_negatives_post_result应该为False
                if 'random_negatives_post_result' in test_cases_result and test_cases_result['random_negatives_post_result']['passed']==True:
                    all_results[d['task_id']]['random_negatives_post_result']=False
                    
            elif test_cases_result['test_case']['run_success']==False:
                # pre_result应该为False
                if test_cases_result['pre_result']['passed']==True:
                    all_results[d['task_id']]['negative_pre_result']=False
        
        all_results[d['task_id']]['pn_pre_result']=all_results[d['task_id']]['positive_pre_result'] and all_results[d['task_id']]['negative_pre_result']
        all_results[d['task_id']]['negative_post_result']=all_results[d['task_id']]['hard_negatives_post_result'] and all_results[d['task_id']]['random_negatives_post_result']
        all_results[d['task_id']]['pn_post_result']=all_results[d['task_id']]['positive_post_result'] and all_results[d['task_id']]['negative_post_result']
        all_results[d['task_id']]['pre_post_result']=all_results[d['task_id']]['pn_pre_result'] and all_results[d['task_id']]['pn_post_result']
        all_results[d['task_id']]['correctness']=all_results[d['task_id']]['positive_pre_result'] and all_results[d['task_id']]['positive_post_result']
        all_results[d['task_id']]['completeness']=all_results[d['task_id']]['negative_pre_result'] and all_results[d['task_id']]['negative_post_result']
        all_results[d['task_id']]['pass']=all_results[d['task_id']]['correctness'] and all_results[d['task_id']]['completeness']
        
    def avg(l):
        NUM_TASKS = 2494
        average = 100* sum(l) / NUM_TASKS
        print(f'avg % = {sum(l)}/{NUM_TASKS} = {average:.2f}')
        return average

    # 提取所有结果列表
    results = {
        # 'positive_pre_result': [r['positive_pre_result'] for r in all_results.values()],
        # 'negative_pre_result': [r['negative_pre_result'] for r in all_results.values()],
        # 'pn_pre_result': [r['pn_pre_result'] for r in all_results.values()],
        # 'positive_post_result': [r['positive_post_result'] for r in all_results.values()],
        # 'hard_negatives_post_results': [r['hard_negatives_post_result'] for r in all_results.values()],
        # 'random_negatives_post_results': [r['random_negatives_post_result'] for r in all_results.values()],
        # 'negative_post_result': [r['negative_post_result'] for r in all_results.values()],
        # 'pn_post_results': [r['pn_post_result'] for r in all_results.values()],
        # 'pre_post_results': [r['pre_post_result'] for r in all_results.values()],
        'correctness': [r['correctness'] for r in all_results.values()],
        'completeness': [r['completeness'] for r in all_results.values()],
        'pass': [r['pass'] for r in all_results.values()]
    }

    # 统一处理并打印结果
    final_results={}
    for name, values in results.items():
        print(f'\n{name}:')
        final_results[name] = avg(values)
    return final_results
    
final_results=eval_spec_results(ds)


打印保存所有模型结果

In [ ]:
from utils import read_jsonl_gz
import pandas as pd

data=[]
# "Qwen3_8B", "Qwen3_14B", "Qwen3_32B", "QwQ_32B"
# "gpt-oss-120b", "gpt-5-mini", "claude-sonnet-4-5-20250929", "gemini-2.5-pro", "gemini-2.5-flash", "deepseek-v3.2-think", "deepseek-v3.2", "gpt-5-chat-latest", "Qwen3_0p6B", "Qwen3_1p7B", "Qwen3_4B"
# "gpt-oss-20b", "Qwen3-0.6B-thinking", "Qwen3-1.7B-thinking", "Qwen3-4B-thinking", "Qwen3-14B-thinking", "Qwen3-32B-thinking","QwQ-32B-thinking"
# 'Qwen3-0.6B', 'Qwen3-1.7B', 'Qwen3-4B', 'Qwen3-8B', 'Qwen3-8B-thinking', 'Qwen3-14B', 'Qwen3-32B', 'QwQ_32B'
model_names = ['Qwen3-0.6B', 'Qwen3-0.6B-thinking', 'Qwen3-1.7B', 'Qwen3-1.7B-thinking', 'Qwen3-4B', 'Qwen3-4B-thinking', 'Qwen3-8B', 'Qwen3-8B-thinking', 'Qwen3-14B', 'Qwen3-14B-thinking', 'Qwen3-32B', 'Qwen3-32B-thinking',               "QwQ-32B-thinking", "gpt-oss-20b", "gpt-oss-120b", "deepseek-v3.2", "gemini-2.5-flash", "gemini-2.5-pro", "gpt-5-mini", "gpt-5-chat-latest", "claude-sonnet-4-5-20250929"]

for model_name in model_names:
    ds=read_jsonl_gz(f'../spec-gen/sample_results/{model_name}.jsonl_results.jsonl.gz')
    print(f'Model: {model_name}')
    print(f'总共测试样例的数量{sum(len(d["test_cases_results"])for d in ds)}')
    final_results=eval_spec_results(ds)
    print('#'*50)
    data.append({'Model': model_name, **final_results})

df = pd.DataFrame(data)
# 保存为Excel文件
df.to_excel('model_results.xlsx', index=False)
    